# Milestone 1 Lab, notebook 1 of 6: The business question

## Where we are

| Notebook | What you do |
|---|---|
| **1. The business question** (you are here) | Read the brief, explore the data, and answer the question by hand |
| 2. Your first LLM calls | Call a model, get a structured reply, and have AI draft the metric definitions |
| 3. Ask the LLM with all the data | Put the data in the prompt, ask the question, and check the answer |
| 4. What is a tool, what is an agent | Build one tool, run the agent loop by hand, then let LangChain run it |
| 5. The agent answers the question | Give an agent six tools and compare it with the single call |
| 6. Verify and go to production | Check an answer with code, and see the production version |

Model requests in this notebook: **0**. OpenRouter's free models allow about 50 a day.

## What this notebook is for

Kestrel Ridge Outfitters is a fictional online retailer of outdoor gear. One of its leaders has a question about profit.

Before we ask any AI, we answer the question ourselves, by hand, with a few lines of pandas. That gives us an **answer key**. In the notebooks that follow, every AI answer is held up against this key, so you can always see whether the AI was right.

There is no AI in this notebook.

> Please run each cell and read the output before moving on.

In [1]:
import pandas as pd
from lab_helpers import (
    LAST_MONTH,
    PREVIOUS_MONTH,
    flags,
    key_row,
    products,
    profit,
    save_result,
    show_doc,
)

`lab_helpers.py` is a small file that sits beside these notebooks. It holds code that isn't about AI (loading the data, displaying documents) and, later, the finished version of things you build, so each notebook can stay short. Open it any time you want to see how something works.

## Step 1: Read what the business is asking for

A real project starts with a requirement, not with code. This milestone has three short design documents in its `docs/` folder. The first is the **BRD** (business requirements document): what the business needs, in business language.

`show_doc` displays a section of a document so you can read it right here.

In [2]:
show_doc("BRD.md", "1. Business problem")

> **From `docs/BRD.md`: 1. Business problem**

Kestrel Ridge Outfitters is a fictional online retailer of outdoor gear. Its leaders can see sales, but they cannot easily see which products actually make money once discounts, refunds, product cost, and delivery costs are taken out. When they do get a profit figure, they cannot tell where it came from or whether to trust it.

They want to ask a profit question in plain language and get an answer that shows its working.

Here is the question. Notice it is really three questions in one:

1. Which products and categories made the most contribution profit?
2. What changed since the month before?
3. How confident should I be in those numbers?

In [3]:
show_doc("BRD.md", "3. The question to answer")

> **From `docs/BRD.md`: 3. The question to answer**

> Which products and categories generated the most contribution profit last month, what changed, and how confident should I be in those numbers?

The analyst must also handle the smaller questions inside it:

- What was a given profit metric for a product or category in a given month?
- How did it change from the month before?
- How is a metric defined?
- Is there a known data problem behind a figure?

"Contribution profit" is the key term. Here is what the business means by it.

In [4]:
show_doc("BRD.md", "7. Profit definitions (business wording)")

> **From `docs/BRD.md`: 7. Profit definitions (business wording)**

- **Net sales:** what customers paid for products in the month, after discounts, less refunds issued in the month.
- **Gross profit:** net sales less what the retailer paid its vendors for the units sold.
- **Contribution profit:** gross profit less the costs that rise with each order: fulfilment, shipping, payment fees, and handling returns.

The exact formulas are in the [LLD](LLD.md).

## Step 2: Meet the data

The data is fictional and prepared by hand. There are 12 products in 3 categories.

In [5]:
products

,product_id,product_name,category
0,P001,Trail water bottle,Hydration
1,P002,Insulated flask,Hydration
2,P003,Hydration pack,Hydration
3,P004,Water filter,Hydration
4,P005,Two-person tent,Camp
5,P006,Sleeping bag,Camp
6,P007,Camp stove,Camp
7,P008,Headlamp,Camp
8,P009,Rain jacket,Apparel
9,P010,Hiking socks,Apparel


The profit table has one row per product per month, for two months. Each row walks down the profit calculation from left to right: sales, then what is taken off, ending at `contribution_profit`.

In [6]:
print(f"{len(profit)} rows. 'Last month' is {LAST_MONTH}; the month before is {PREVIOUS_MONTH}.")
profit.head(3)

24 rows. 'Last month' is 2026-10; the month before is 2026-09.


,month,product_id,units_sold,gross_sales,discounts,refunds,net_sales,product_cost,gross_profit,fulfillment_cost,shipping_cost,payment_fees,return_cost,contribution_profit,metric_version
0,2026-09,P001,420,16800.0,840.0,76.0,15884.0,7560.0,8324.0,1050.0,1680.0,478.80,12.0,5103.20,1.0.0
1,2026-09,P002,260,9100.0,455.0,66.5,8578.5,3900.0,4678.5,650.0,1040.0,259.35,12.0,2717.15,1.0.0
2,2026-09,P003,95,8550.0,855.0,162.0,7533.0,4560.0,2973.0,285.0,570.0,230.85,16.0,1871.15,1.0.0


## Step 3: Answer the question by hand

### 3a. Which products made the most contribution profit last month?

Keep last month's rows, add the product names, and sort.

In [7]:
last_month = profit[profit["month"] == LAST_MONTH].merge(products, on="product_id")
ranked = last_month.sort_values("contribution_profit", ascending=False)

ranked[["product_id", "product_name", "category", "contribution_profit"]]

,product_id,product_name,category,contribution_profit
0,P001,Trail water bottle,Hydration,5188.20
7,P008,Headlamp,Camp,4250.00
8,P009,Rain jacket,Apparel,3841.00
5,P006,Sleeping bag,Camp,3297.30
3,P004,Water filter,Hydration,3129.11
11,P012,Fleece pullover,Apparel,2934.06
1,P002,Insulated flask,Hydration,2541.35
2,P003,Hydration pack,Hydration,2499.40
9,P010,Hiking socks,Apparel,2464.32
4,P005,Two-person tent,Camp,1818.96


The top three are the trail water bottle, the headlamp, and the rain jacket.

Now look at the last row. The camp stove shows `NaN`, which means "no value". Its product cost is missing from the data, so its profit cannot be calculated. The business rule (BRD section 6) is that a figure like this is reported as **unavailable**. It is never treated as zero and never estimated.

### 3b. Which categories made the most?

Add up the products in each category.

In [8]:
by_category = last_month.groupby("category")["contribution_profit"].sum().round(2).sort_values(ascending=False)
by_category

category
Hydration    13358.06
Apparel       9746.38
Camp          9366.26
Name: contribution_profit, dtype: float64

Hydration is the top category.

### 3c. What changed since the month before?

Put the two months side by side and subtract.

In [9]:
changes = profit.merge(products, on="product_id").pivot_table(index="category", columns="month", values="contribution_profit", aggfunc="sum")
changes["change"] = (changes[LAST_MONTH] - changes[PREVIOUS_MONTH]).round(2)
changes

month,2026-09,2026-10,change
category,,,
Apparel,15661.25,9746.38,-5914.87
Camp,11961.49,9366.26,-2595.23
Hydration,12982.00,13358.06,376.06


Hydration rose a little. Apparel fell a lot.

**Be careful with Camp.** It appears to have fallen by about 2,600. But Camp's October total leaves out the camp stove (its profit is unavailable), while September's total includes it. We are comparing three products with four. That "fall" is not real, so the honest answer for Camp is **cannot be compared**.

This is a trap. Remember it: we will see whether the AI falls into it.

### 3d. How confident should we be?

The known problems with the data are recorded as quality flags.

In [10]:
pd.set_option("display.max_colwidth", None)
flags[["flag_id", "month", "product_id", "severity", "description"]]

,flag_id,month,product_id,severity,description
0,QF-001,2026-10,P007,high,"No product cost is recorded for Camp stove in 2026-10, so its gross profit and contribution profit cannot be calculated. The Camp category total leaves it out."
1,QF-002,2026-10,P001,medium,Three orders for Trail water bottle appear twice in 2026-10. Its sales and profit may be overstated.
2,QF-003,2026-09,P011,low,"A 2026-09 refund for Sun hat is larger than the order it refunds. Its September net sales may be understated, which affects its change from September to October."


Two flags affect last month: the camp stove's missing cost (high severity) and duplicated orders for the trail water bottle, our number one product (medium severity).

The rule for confidence is simple: the worst flag decides. A high severity flag means **low** confidence, a medium one means medium, and otherwise confidence is high. So our confidence in last month's answer is **low**.

## Step 4: Write down the answer key

We now know the correct answer. Let's write it down as a small table, one line per fact. `key_row` just packs a line into a standard form.

In [11]:
figures = []
for place, row in enumerate(ranked.head(3).itertuples(), start=1):
    figures.append(key_row(f"#{place} product: {row.product_name}", row.product_id, row.contribution_profit))

figures.append(key_row("Top category: Hydration", "Hydration", by_category["Hydration"]))
figures.append(key_row("Hydration, change since the month before", "Hydration", changes.loc["Hydration", "change"], kind="change"))
figures.append(key_row("Apparel, change since the month before (the biggest fall)", "Apparel", changes.loc["Apparel", "change"], kind="change"))
figures.append(key_row("Camp, change since the month before (cannot be compared)", "Camp", None, kind="change"))
figures.append(key_row("Camp stove (its cost is missing)", "P007", None))

answer_key = {"figures": figures, "confidence": "low", "flags": ["QF-001", "QF-002"]}

pd.DataFrame(figures)[["what", "entity_id", "kind", "value"]]

,what,entity_id,kind,value
0,#1 product: Trail water bottle,P001,value,5188.20
1,#2 product: Headlamp,P008,value,4250.00
2,#3 product: Rain jacket,P009,value,3841.00
3,Top category: Hydration,Hydration,value,13358.06
4,"Hydration, change since the month before",Hydration,change,376.06
5,"Apparel, change since the month before (the biggest fall)",Apparel,change,-5914.87
6,"Camp, change since the month before (cannot be compared)",Camp,change,NaN
7,Camp stove (its cost is missing),P007,value,NaN


`value` is empty for the last two lines on purpose: the correct answer there is "unavailable".

Save the key so the next notebooks can load it.

In [12]:
save_result("answer_key", answer_key)

Saved results/answer_key.json


## What we now know

The correct answer to the leader's question, in plain words:

- **Top products:** trail water bottle ($5,188.20), headlamp ($4,250.00), rain jacket ($3,841.00).
- **Top category:** Hydration ($13,358.06), up $376.06 on the month before.
- **What changed:** Apparel fell by $5,914.87. Camp cannot be compared, because the camp stove's cost is missing.
- **Confidence:** low, because of two known data problems (QF-001 and QF-002).

It took us about ten lines of pandas, and we had to know pandas, know the table, and notice the Camp trap. A retail leader can't do that. They want to ask in plain language.

**Next: notebook 2**, where we make our first calls to a language model.